# 🏥 VoiceScribe AI - 100% Free Cloud GPU (NVIDIA T4 16GB)
Run ambient medical AI documentation with **Whisper Large-v3-Turbo + Qwen 2.5 7B** in the cloud with zero credit card required.

👉 **Make sure GPU is enabled**: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
# [CELL 1] System Dependencies & Cloudflare Tunnel
!curl -fsSL https://ollama.com/install.sh | sh
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y -q nodejs
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
print('✅ Dependencies and Cloudflare Tunnel installed!')

In [ ]:
# [CELL 2] Start Ollama & Download Qwen 2.5 7B on T4 GPU
import subprocess, time
print('[1/2] Starting Ollama background server...')
subprocess.Popen(['ollama', 'serve'])
time.sleep(3)
print('[2/2] Downloading Qwen 2.5 7B (Clinical Multilingual Champion)...')
!ollama pull qwen2.5:7b
print('✅ Ollama is ready on GPU!')

In [ ]:
# [CELL 3] Clone Repo & Install Backend Packages
!rm -rf VoiceScribeOffline
!git clone https://github.com/saishrikarmk24/VoiceScribeOffline.git
%cd VoiceScribeOffline/backend
!pip install -q -r requirements.txt
!pip install -q faster-whisper

import os
os.environ['AI_MODE'] = 'local'
os.environ['LOCAL_LLM_BASE_URL'] = 'http://localhost:11434/v1'
os.environ['LOCAL_LLM_MODEL'] = 'qwen2.5:7b'
os.environ['ASR_PROVIDER'] = 'indic_whisper'
os.environ['FASTER_WHISPER_MODEL'] = 'large-v3-turbo'
os.environ['DEVICE'] = 'cuda'
print('✅ Backend configured for CUDA GPU!')

In [ ]:
# [CELL 4] Build React Frontend Assets
%cd /content/VoiceScribeOffline/frontend
!npm install --silent
!npm run build
print('✅ Frontend built successfully!')

In [ ]:
# [CELL 5] Launch Unified Server (Web UI + API) & Generate Shareable Link
import subprocess, re, time

# Start FastAPI backend on port 8000 (which directly mounts and serves the built frontend)
%cd /content/VoiceScribeOffline/backend
backend_proc = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
print('🚀 VoiceScribe Server starting on port 8000...')
time.sleep(3)

# Open secure Cloudflare tunnel to port 8000
print('🌐 Generating secure public HTTPS tunnel...')
cf_tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

public_url = None
for line in cf_tunnel.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print('\n' + '='*70)
print('🎉 VOICESCRIBE AI IS LIVE ON CLOUD GPU!')
print('='*70)
print(f'\n👉 SHAREABLE WEB APP LINK:  {public_url}\n')
print('Doctor Login:')
print('  • Doctor ID or Email: saksham@sims.com (or DOC-101)')
print('  • Password:           Doctor@1234')
print('\nAdmin Login:')
print('  • Admin Email:        admin@sims.com')
print('  • Password:           Admin@1234')
print('='*70)

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('Stopping server...')
    backend_proc.terminate()
    cf_tunnel.terminate()
